# 1. Preparação Geral dos Dados e Constantes

Nesta etapa, carregamos a base limpa do NHIS 2025 (`adult25_limpo.csv`), definimos o `ID_DOMICILIO` como índice e isolamos a variável alvo multiclasse (`SEVERIDADE_DEPRESSAO_PHQ`), filtrando os registros indefinidos na Target (valores `-1` e `5`).

Além disso, isolamos a coluna `PESO_AMOSTRAL` e removemos do conjunto de preditores ($X$) todas as colunas de sintomas diretos de depressão para evitar vazamento de dados (*data leakage*).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==============================================================================
# CONFIGURAÇÃO DE VARIÁVEIS GLOBAIS / CONSTANTES
# ==============================================================================
ID_COLUNA = 'ID_DOMICILIO'
PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET = 'SEVERIDADE_DEPRESSAO_PHQ'

COLUNAS_SINTOMAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM',
    'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM',
    'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM',
    'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM',
    'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO',
    'REMEDIO_DEPRESSAO',
    'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO'
]

# 1. Carregar a base limpa e definir o ID como índice
df = pd.read_csv('adult25_limpo.csv').set_index(ID_COLUNA)

# 2. Eliminar os registros indefinidos no TARGET (-1 e 5)
df = df[~df[TARGET].isin([-1, 5])].copy()

# 3. Isolar Target (y), Peso Amostral e Features Preditoras (X)
y = df[TARGET]
sample_weights = df[PESO_COLUNA]
colunas_remover = [TARGET, PESO_COLUNA] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[col for col in colunas_remover if col in df.columns])

# 2. Cenário A: Treinamento Mantendo Códigos Indeterminados (-1) nas Features

Neste primeiro cenário, mantemos as categorias de não-resposta (`-1`) presentes nas variáveis preditoras ($X$), permitindo que o modelo de árvore e a codificação One-Hot tratem o valor como uma categoria válida.

In [2]:
# Split Estratificado
X_train_A, X_test_A, y_train_A, y_test_A, w_train_A, w_test_A = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=42, stratify=y
)

# Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [col for col in X_train_A.columns if col not in num_cols]

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_A = preprocessor_A.fit_transform(X_train_A)
X_test_proc_A = preprocessor_A.transform(X_test_A)

# Modelo Random Forest
model_A = RandomForestClassifier(random_state=42, class_weight='balanced')
model_A.fit(X_train_proc_A, y_train_A, sample_weight=w_train_A)

# Avaliação Cenário A
y_pred_A = model_A.predict(X_test_proc_A)
y_proba_A = model_A.predict_proba(X_test_proc_A)

print("--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---")
print(classification_report(y_test_A, y_pred_A, sample_weight=w_test_A, zero_division=0))

print("--- CENÁRIO A: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_A, y_pred_A, sample_weight=w_test_A)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_A, y_proba_A, multi_class='ovr', sample_weight=w_test_A):.4f}")

--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 40224045.76199999
           2       0.44      0.61      0.51 7209618.388000002
           3       0.44      0.25      0.32 2466480.354
           4       0.63      0.64      0.63 1334503.7569999998

    accuracy                           0.82 51234648.26099999
   macro avg       0.61      0.60      0.59 51234648.26099999
weighted avg       0.83      0.82      0.82 51234648.26099999

--- CENÁRIO A: Matriz de Confusão 4x4 ---
[[35900034  4173661   128854    21497]
 [ 2268874  4405756   425006   109983]
 [  142188  1321019   627418   375855]
 [   29742   198955   256571   849235]]
ROC-AUC Score (OvR): 0.9140


# 3. Cenário B: Treinamento Apenas com Informações Completas (Remoção de -1 no X)

Conforme discutido com a equipe e orientação do professor, realizamos um teste estrito filtrando qualquer observação que contenha não-resposta (`-1`) em qualquer feature preditora. A amostragem retida foi de **22.084 registros (93.5% dos dados)**.

In [3]:
# Filtragem Estrita das Features
mascara_sem_menos1 = (X != -1).all(axis=1)
X_strict = X[mascara_sem_menos1].copy()
y_strict = y[mascara_sem_menos1].copy()
w_strict = sample_weights[mascara_sem_menos1].copy()

print(f"Total de registros na base completa estrita: {len(X_strict)}")

# Split Estratificado Cenário B
X_train_B, X_test_B, y_train_B, y_test_B, w_train_B, w_test_B = train_test_split(
    X_strict, y_strict, w_strict, test_size=0.2, random_state=42, stratify=y_strict
)

# Pré-processamento
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_B = preprocessor_B.fit_transform(X_train_B)
X_test_proc_B = preprocessor_B.transform(X_test_B)

# Modelo Random Forest Cenário B
model_B = RandomForestClassifier(random_state=42, class_weight='balanced')
model_B.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

# Avaliação Cenário B
y_pred_B = model_B.predict(X_test_proc_B)
y_proba_B = model_B.predict_proba(X_test_proc_B)

print("--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---")
print(classification_report(y_test_B, y_pred_B, sample_weight=w_test_B, zero_division=0))

print("--- CENÁRIO B: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_B, y_pred_B, sample_weight=w_test_B)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_B, y_proba_B, multi_class='ovr', sample_weight=w_test_B):.4f}")

Total de registros na base completa estrita: 22084
--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 37773204.54699994
           2       0.42      0.60      0.49 6824341.566000004
           3       0.28      0.19      0.23 2187297.4779999997
           4       0.62      0.52      0.57 1212457.585

    accuracy                           0.81 47997301.17599995
   macro avg       0.56      0.55      0.55 47997301.17599995
weighted avg       0.83      0.81      0.81 47997301.17599995

--- CENÁRIO B: Matriz de Confusão 4x4 ---
[[33571356  3965806   230880     5162]
 [ 2156776  4069561   515537    82468]
 [  116638  1355470   420541   294648]
 [       0   233664   349171   629622]]
ROC-AUC Score (OvR): 0.9055


# 4. Definição do Espaço de Busca de Hiperparâmetros Refinado

Definimos uma grade estendida de parâmetros para otimizar o desempenho do Random Forest tanto no cenário multinível quanto no binário.

In [4]:
param_dist_refinado = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [20, 25, 30, 35, None],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', 0.2, 0.3],
    'class_weight': ['balanced', 'balanced_subsample']
}

# 5. Otimização do Modelo Multinível (4 Classes) com Hiperparâmetros Refinados

Aplicamos o `RandomizedSearchCV` focado no $F1\text{-macro}$ para melhorar a discriminação dos 4 níveis do PHQ-9.

In [5]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Configurar o RandomizedSearchCV no Cenário B
random_search_multi = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_refinado,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 2. Treinamento
random_search_multi.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (MULTINÍVEL) ---")
print(random_search_multi.best_params_)

# 3. Avaliação no Conjunto de Teste
best_model_multi = random_search_multi.best_estimator_
y_pred_tuned = best_model_multi.predict(X_test_proc_B)
y_proba_tuned = best_model_multi.predict_proba(X_test_proc_B)

print("\n--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---")
print(classification_report(y_test_B, y_pred_tuned, sample_weight=w_test_B, zero_division=0))

print("--- MATRIZ DE CONFUSÃO DA AMOSTRA (4x4 - Contagem Bruta de Linhas) ---")
print(confusion_matrix(y_test_B, y_pred_tuned))

print("\n--- MATRIZ DE CONFUSÃO POPULACIONAL (4x4 - Projeção Expandida) ---")
print(np.round(confusion_matrix(y_test_B, y_pred_tuned, sample_weight=w_test_B)).astype(int))

print(f"\nROC-AUC Score (OvR) Otimizado: {roc_auc_score(y_test_B, y_proba_tuned, multi_class='ovr', sample_weight=w_test_B):.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (MULTINÍVEL) ---
{'n_estimators': 400, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.2, 'max_depth': 30, 'class_weight': 'balanced_subsample'}

--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---
              precision    recall  f1-score   support

           1       0.92      0.92      0.92 37773204.54699994
           2       0.45      0.49      0.47 6824341.566000004
           3       0.32      0.26      0.29 2187297.4779999997
           4       0.64      0.66      0.65 1212457.585

    accuracy                           0.82 47997301.17599995
   macro avg       0.58      0.58      0.58 47997301.17599995
weighted avg       0.82      0.82      0.82 47997301.17599995

--- MATRIZ DE CONFUSÃO DA AMOSTRA (4x4 - Contagem Bruta de Linhas) ---
[[3208  255   17    2]
 [ 251  307   60   10]
 [  24   95   56   27]
 [   1   10   25   69]]

--- MATRIZ DE CONFUSÃO POPULA

# 6. Verificação e Distribuição do Tamanho dos Grupos

Análise comparativa do número absoluto de registos na amostra e a projeção populacional expandida.

In [6]:
mapa_niveis = {
    1: 'Nível 1: Mínimo / Sem Depressão',
    2: 'Nível 2: Leve',
    3: 'Nível 3: Moderado',
    4: 'Nível 4: Grave'
}

contagem_amostral = df['SEVERIDADE_DEPRESSAO_PHQ'].value_counts().sort_index()
projecao_populacional = df.groupby('SEVERIDADE_DEPRESSAO_PHQ')['PESO_AMOSTRAL'].sum()

tabela_grupos_total = pd.DataFrame({
    'Descrição do Nível': contagem_amostral.index.map(mapa_niveis),
    'Amostra Bruta (Linhas)': contagem_amostral.values,
    '% Amostra': (contagem_amostral.values / len(df)) * 100,
    'População Expandida': projecao_populacional.values,
    '% População': (projecao_populacional.values / df['PESO_AMOSTRAL'].sum()) * 100
})

print("--- 1. TAMANHO DOS GRUPOS NA BASE TOTAL (22.084 REGISTROS ESTRITOS) ---")
print(tabela_grupos_total.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n" + "="*80 + "\n")
print("--- 2. DISTRIBUIÇÃO DOS GRUPOS NO SPLIT ESTRATIFICADO (TREINO / TESTE) ---")

tr_counts = y_train_B.value_counts().sort_index()
tr_pop = y_train_B.to_frame().join(w_train_B).groupby('SEVERIDADE_DEPRESSAO_PHQ')['PESO_AMOSTRAL'].sum()

te_counts = y_test_B.value_counts().sort_index()
te_pop = y_test_B.to_frame().join(w_test_B).groupby('SEVERIDADE_DEPRESSAO_PHQ')['PESO_AMOSTRAL'].sum()

tabela_split = pd.DataFrame({
    'Nível': tr_counts.index,
    'Treino (Linhas)': tr_counts.values,
    '% Treino': (tr_counts.values / len(y_train_B)) * 100,
    'Teste (Linhas)': te_counts.values,
    '% Teste': (te_counts.values / len(y_test_B)) * 100,
    'População Teste': te_pop.values
})

print(tabela_split.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

--- 1. TAMANHO DOS GRUPOS NA BASE TOTAL (22.084 REGISTROS ESTRITOS) ---
             Descrição do Nível  Amostra Bruta (Linhas)  % Amostra  População Expandida  % População
Nível 1: Mínimo / Sem Depressão                   18604      78.77         202918756.60        78.58
                  Nível 2: Leve                    3352      14.19          36966381.86        14.31
              Nível 3: Moderado                    1083       4.59          11888776.45         4.60
                 Nível 4: Grave                     580       2.46           6466187.57         2.50


--- 2. DISTRIBUIÇÃO DOS GRUPOS NO SPLIT ESTRATIFICADO (TREINO / TESTE) ---
 Nível  Treino (Linhas)  % Treino  Teste (Linhas)  % Teste  População Teste
     1            13925     78.82            3482    78.83      37773204.55
     2             2511     14.21             628    14.22       6824341.57
     3              810      4.58             202     4.57       2187297.48
     4              421      2.38         

# 7. Otimização e Avaliação do Modelo Binário (0 = Sem Depressão, 1 = Com Depressão)

Execução da otimização com a grade refinada para a abordagem binária de rastreio.

In [7]:
# 1. Carregar a base limpa e aplicar a transformação binária
df_bin = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')
df_bin = df_bin[~df_bin['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df_bin['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df_bin['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS_BIN = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# 2. Isolar X, y e Pesos nas Features Completas
y_bin = df_bin[TARGET_BIN]
weights_bin = df_bin[PESO_COLUNA]
X_base_bin = df_bin.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS_BIN if col in df_bin.columns])

mascara_completos = (X_base_bin != -1).all(axis=1)
X_bin = X_base_bin[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights_bin[mascara_completos].copy()

# 3. Split Estratificado Binário
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor_bin = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor_bin.fit_transform(X_tr)
X_te_proc = preprocessor_bin.transform(X_te)

# 5. RandomizedSearchCV Binário
random_search_bin = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_refinado,
    n_iter=20,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando a busca de hiperparâmetros binários (RandomizedSearchCV)...")
random_search_bin.fit(X_tr_proc, y_tr, sample_weight=w_tr)

rf_binario_otimizado = random_search_bin.best_estimator_

print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (BINÁRIO) ---")
for param, val in random_search_bin.best_params_.items():
    print(f"{param}: {val}")

# 6. Avaliação
y_pred_bin = rf_binario_otimizado.predict(X_te_proc)
y_proba_bin = rf_binario_otimizado.predict_proba(X_te_proc)[:, 1]

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (PONDERADO POR PESOS POPULACIONAIS) ---")
print(classification_report(y_te, y_pred_bin, sample_weight=w_te, zero_division=0))

print("\n--- MATRIZ DE CONFUSÃO DA AMOSTRA (2x2 - Contagem Bruta de Linhas) ---")
print(confusion_matrix(y_te, y_pred_bin))

print("\n--- MATRIZ DE CONFUSÃO POPULACIONAL (2x2 - Projeção Expandida) ---")
print(np.round(confusion_matrix(y_te, y_pred_bin, sample_weight=w_te)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba_bin, sample_weight=w_te):.4f}")

Iniciando a busca de hiperparâmetros binários (RandomizedSearchCV)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (BINÁRIO) ---
n_estimators: 400
min_samples_split: 4
min_samples_leaf: 3
max_features: sqrt
max_depth: 25
class_weight: balanced_subsample

--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (PONDERADO POR PESOS POPULACIONAIS) ---
              precision    recall  f1-score   support

           0       0.92      0.90      0.91 37773204.546999976
           1       0.67      0.72      0.70 10425277.420000015

    accuracy                           0.86 48198481.96699999
   macro avg       0.80      0.81      0.80 48198481.96699999
weighted avg       0.87      0.86      0.87 48198481.96699999


--- MATRIZ DE CONFUSÃO DA AMOSTRA (2x2 - Contagem Bruta de Linhas) ---
[[3159  323]
 [ 264  671]]

--- MATRIZ DE CONFUSÃO POPULACIONAL (2x2 - Projeção Expandida) ---
[[34113608  3659597]
 [ 2911488  7513789]]

ROC-AUC Score (Binário): 0.9